# Predictive Maintenance: NASA CMAPSS Turbofan Engine Analysis

**Dataset**: NASA CMAPSS (Commercial Modular Aero-Propulsion System Simulation) FD001

**Source**: [NASA Prognostics Center of Excellence](https://ti.arc.nasa.gov/tech/dash/groups/pcoe/prognostic-data-repository/)

**Citation**: A. Saxena, K. Goebel, D. Simon, and N. Eklund, "Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation," in Proceedings of the 1st International Conference on Prognostics and Health Management (PHM08), Denver CO, Oct 2008.

---

This notebook demonstrates:
1. **Data exploration** of real turbofan engine sensor readings
2. **Feature engineering** with rolling windows and normalization
3. **RUL prediction** using Random Forest and XGBoost
4. **Model evaluation** with MAE, RMSE metrics
5. **Visualizations** of sensor degradation curves and predictions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ML imports
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('Libraries loaded successfully')

## 1. Data Loading

The CMAPSS FD001 dataset contains:
- **100 training engines**: run-to-failure trajectories
- **100 test engines**: truncated before failure
- **26 columns**: unit ID, cycle, 3 operating settings, 21 sensor measurements
- **RUL_FD001.txt**: true Remaining Useful Life for test engines

All data is **real NASA simulation data** — no synthetic generation.

In [ ]:
# Define column names (from NASA documentation)
cols = ['unit_number', 'time_in_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + [f'sensor_{i}' for i in range(1, 22)]

# Load training data
train_path = Path('../data/raw/train_FD001.txt')
train_df = pd.read_csv(train_path, sep='\s+', header=None, names=cols)

# Load test data
test_path = Path('../data/raw/test_FD001.txt')
test_df = pd.read_csv(test_path, sep='\s+', header=None, names=cols)

# Load RUL labels
rul_path = Path('../data/raw/RUL_FD001.txt')
true_rul = pd.read_csv(rul_path, header=None, names=['RUL']).values.flatten()

print(f"Training data: {train_df.shape[0]:,} rows, {train_df.shape[1]} columns")
print(f"Test data: {test_df.shape[0]:,} rows, {test_df.shape[1]} columns")
print(f"True RUL values: {len(true_rul)} engines")
print(f"\nTraining engines: {train_df['unit_number'].nunique()}")
print(f"Test engines: {test_df['unit_number'].nunique()}")

train_df.head()

In [ ]:
# Quick data profile
print('=== TRAINING DATA PROFILE ===')
print(train_df.describe().T.round(2).to_string())
print('\n=== TEST DATA PROFILE ===')
print(test_df.describe().T.round(2).to_string())

## 2. Compute Remaining Useful Life (RUL)

For training data: RUL = max_cycle - current_cycle for each engine

For test data: RUL is provided in RUL_FD001.txt

In [ ]:
# Compute RUL for training data
max_cycles = train_df.groupby('unit_number')['time_in_cycles'].max().reset_index()
max_cycles.columns = ['unit_number', 'max_cycle']
train_df = train_df.merge(max_cycles, on='unit_number')
train_df['RUL'] = train_df['max_cycle'] - train_df['time_in_cycles']
train_df = train_df.drop('max_cycle', axis=1)

print(f"Training RUL range: {train_df['RUL'].min()} to {train_df['RUL'].max()} cycles")
print(f"Mean training RUL: {train_df['RUL'].mean():.1f} cycles")

# Distribution of max cycles (engine lifetimes)
engine_lifetimes = train_df.groupby('unit_number')['time_in_cycles'].max()
print(f"\nEngine lifetime range: {engine_lifetimes.min()} to {engine_lifetimes.max()} cycles")
print(f"Mean engine lifetime: {engine_lifetimes.mean():.1f} cycles")

# RUL distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['RUL'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Remaining Useful Life (cycles)')
axes[0].set_ylabel('Count')
axes[0].set_title('Training RUL Distribution')

axes[1].hist(engine_lifetimes, bins=20, color='coral', edgecolor='white')
axes[1].set_xlabel('Engine Lifetime (cycles)')
axes[1].set_ylabel('Count')
axes[1].set_title('Engine Failure Time Distribution')

plt.tight_layout()
plt.savefig('../figures/rul_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved: ../figures/rul_distributions.png')

## 3. Sensor Data Exploration

Visualize how sensor readings degrade over time for sample engines.

In [ ]:
# Select 4 representative engines with different lifetimes
sample_engines = [1, 25, 50, 75]
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]

fig, axes = plt.subplots(3, 7, figsize=(20, 10))
axes = axes.flatten()

for idx, sensor in enumerate(sensor_cols):
    ax = axes[idx]
    for engine in sample_engines:
        engine_data = train_df[train_df['unit_number'] == engine]
        ax.plot(engine_data['time_in_cycles'], engine_data[sensor], alpha=0.7, label=f'Engine {engine}')
    ax.set_title(sensor)
    ax.set_xlabel('Cycle')
    ax.tick_params(labelsize=8)
    if idx == 0:
        ax.legend(fontsize=7, loc='upper left')

plt.suptitle('Sensor Degradation Curves (4 Sample Engines)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/sensor_degradation_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../figures/sensor_degradation_curves.png')

In [ ]:
# Identify constant sensors (no information value)
sensor_variance = train_df[sensor_cols].var().sort_values()
print('Sensor variance (ascending):')
print(sensor_variance.round(4).to_string())

# Drop sensors with near-zero variance
constant_sensors = sensor_variance[sensor_variance < 0.01].index.tolist()
useful_sensors = [s for s in sensor_cols if s not in constant_sensors]
print(f"\nConstant sensors dropped: {constant_sensors}")
print(f"Useful sensors retained: {len(useful_sensors)} / {len(sensor_cols)}")

## 4. Feature Engineering

Create time-windowed features for RUL prediction:
- Rolling mean / std over last N cycles
- Operating condition normalization
- Time-based features

In [ ]:
def create_features(df, sensors, window=10):
    """
    Create rolling-window features for each engine.
    
    Args:
        df: DataFrame with unit_number, time_in_cycles, sensors
        sensors: List of sensor column names to use
        window: Rolling window size
    
    Returns:
        DataFrame with original + engineered features
    """
    feature_df = df.copy()
    
    for sensor in sensors:
        # Rolling statistics per engine
        feature_df[f'{sensor}_rolling_mean'] = (
            feature_df.groupby('unit_number')[sensor]
            .rolling(window=window, min_periods=1)
            .mean().reset_index(level=0, drop=True)
        )
        feature_df[f'{sensor}_rolling_std'] = (
            feature_df.groupby('unit_number')[sensor]
            .rolling(window=window, min_periods=1)
            .std().reset_index(level=0, drop=True)
        ).fillna(0)
        
        # Rate of change
        feature_df[f'{sensor}_diff'] = (
            feature_df.groupby('unit_number')[sensor]
            .diff().fillna(0)
        )
    
    # Time-based features
    feature_df['cycle_norm'] = feature_df.groupby('unit_number')['time_in_cycles'].transform(
        lambda x: x / x.max()
    )
    
    # Operating condition cluster (FD001 has 1 condition, but keep for generality)
    feature_df['op_setting_sum'] = (
        feature_df['op_setting_1'] + feature_df['op_setting_2'] + feature_df['op_setting_3']
    )
    
    return feature_df

# Apply feature engineering
print('Engineering features for training data...')
train_features = create_features(train_df, useful_sensors, window=10)

print(f"Features before: {len(train_df.columns)}")
print(f"Features after: {len(train_features.columns)}")

print('\nEngineering features for test data...')
test_features = create_features(test_df, useful_sensors, window=10)
print(f"Test features: {len(test_features.columns)}")

In [ ]:
# Prepare feature matrix
feature_cols = [c for c in train_features.columns if c not in ['unit_number', 'time_in_cycles', 'RUL'] and (not c.startswith('sensor_') or c in useful_sensors)]

# Actually, let us be precise: include original useful sensors + engineered features
engineered_cols = [c for c in train_features.columns if '_rolling_mean' in c or '_rolling_std' in c or '_diff' in c or c in ['cycle_norm', 'op_setting_sum']]
feature_cols = useful_sensors + engineered_cols

X_train = train_features[feature_cols].fillna(0)
y_train = train_features['RUL']

# For test data, take the last cycle of each engine (predict on final known state)
test_last = test_features.groupby('unit_number').last().reset_index()
X_test = test_last[feature_cols].fillna(0)

print(f"Training matrix: {X_train.shape}")
print(f"Test matrix: {X_test.shape}")
print(f"Feature names ({len(feature_cols)}):")
for f in feature_cols:
    print(f'  {f}')

## 5. RUL Prediction Models

Train two models:
1. **Random Forest Regressor**
2. **XGBoost Regressor**

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Features standardized')
print(f"Train mean: {X_train_scaled.mean(axis=0).mean():.4f}, std: {X_train_scaled.std(axis=0).mean():.4f}")

In [ ]:
# Model 1: Random Forest
print('Training Random Forest...')
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

# Model 2: XGBoost
print('Training XGBoost...')
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)
xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)

print('\nModels trained successfully')

## 6. Model Evaluation

Compare predicted vs. actual RUL for both models.

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n=== {name} ===")
    print(f"  MAE:  {mae:.2f} cycles")
    print(f"  RMSE: {rmse:.2f} cycles")
    print(f"  R2:   {r2:.3f}")
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

rf_metrics = evaluate_model('Random Forest', true_rul, rf_pred)
xgb_metrics = evaluate_model('XGBoost', true_rul, xgb_pred)

# Summary table
metrics_df = pd.DataFrame([
    {'Model': 'Random Forest', **rf_metrics},
    {'Model': 'XGBoost', **xgb_metrics}
])
print('\n=== Comparison ===')
print(metrics_df.round(2).to_string(index=False))

In [ ]:
# Predicted vs Actual RUL scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (name, pred) in enumerate([('Random Forest', rf_pred), ('XGBoost', xgb_pred)]):
    ax = axes[idx]
    ax.scatter(true_rul, pred, alpha=0.6, c='steelblue', edgecolors='white', s=60)
    
    # Perfect prediction line
    max_val = max(true_rul.max(), pred.max())
    ax.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect prediction')
    
    ax.set_xlabel('Actual RUL (cycles)')
    ax.set_ylabel('Predicted RUL (cycles)')
    ax.set_title(f"{name}\nMAE={metrics_df[metrics_df.Model==name].MAE.iloc[0]:.1f}, RMSE={metrics_df[metrics_df.Model==name].RMSE.iloc[0]:.1f}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Actual RUL — NASA CMAPSS FD001', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/predicted_vs_actual_rul.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../figures/predicted_vs_actual_rul.png')

In [ ]:
# Residual analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (name, pred) in enumerate([('Random Forest', rf_pred), ('XGBoost', xgb_pred)]):
    residuals = true_rul - pred
    ax = axes[idx]
    ax.hist(residuals, bins=25, color='coral', edgecolor='white')
    ax.axvline(x=0, color='black', linestyle='--', lw=2)
    ax.set_xlabel('Residual (Actual - Predicted)')
    ax.set_ylabel('Count')
    ax.set_title(f"{name} Residuals\nMean={residuals.mean():.1f}, Std={residuals.std():.1f}")

plt.suptitle('RUL Prediction Residuals', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/rul_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../figures/rul_residuals.png')

## 7. Feature Importance

Which sensor features drive RUL prediction?

In [ ]:
# Feature importance from XGBoost (typically most informative)
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True)

# Top 20 features
top_n = 20
top_features = importance.tail(top_n)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(range(len(top_features)), top_features['importance'], color='teal')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'], fontsize=9)
ax.set_xlabel('Feature Importance (XGBoost)')
ax.set_title(f'Top {top_n} Features for RUL Prediction — NASA CMAPSS FD001')

plt.tight_layout()
plt.savefig('../figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../figures/feature_importance.png')

## 8. Degradation Trajectory Visualization

Show how actual and predicted RUL evolve for sample test engines.

In [ ]:
# For visualization, we need per-cycle predictions on test data
# Predict RUL for every cycle in test set
X_test_all = test_features[feature_cols].fillna(0)
X_test_all_scaled = scaler.transform(X_test_all)
test_features['RUL_pred'] = xgb_model.predict(X_test_all_scaled)

# Plot degradation curves for 4 test engines
sample_test_engines = [1, 25, 50, 75]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, engine in enumerate(sample_test_engines):
    ax = axes[idx]
    engine_data = test_features[test_features['unit_number'] == engine]
    
    # Predicted RUL trajectory
    ax.plot(engine_data['time_in_cycles'], engine_data['RUL_pred'], 
            'b-', lw=2, label='Predicted RUL')
    
    # True final RUL (horizontal line from last cycle)
    true_final = true_rul[engine - 1]
    ax.axhline(y=true_final, color='r', linestyle='--', lw=2, label=f'True final RUL = {true_final}')
    
    # Last cycle marker
    last_cycle = engine_data['time_in_cycles'].max()
    last_pred = engine_data['RUL_pred'].iloc[-1]
    ax.scatter([last_cycle], [last_pred], color='blue', s=100, zorder=5)
    
    ax.set_xlabel('Cycle')
    ax.set_ylabel('RUL (cycles)')
    ax.set_title(f'Test Engine {engine}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('RUL Degradation Trajectories — XGBoost Predictions', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/rul_degradation_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../figures/rul_degradation_trajectories.png')

## 9. Correlation Analysis

Sensor correlation with RUL reveals which measurements most directly indicate degradation.

In [ ]:
# Compute correlation of each sensor with RUL
sensor_rul_corr = train_df[useful_sensors + ['RUL']].corr()['RUL'].drop('RUL').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['darkgreen' if c > 0 else 'darkred' for c in sensor_rul_corr.values]
ax.barh(range(len(sensor_rul_corr)), sensor_rul_corr.values, color=colors)
ax.set_yticks(range(len(sensor_rul_corr)))
ax.set_yticklabels(sensor_rul_corr.index, fontsize=10)
ax.set_xlabel('Correlation with RUL')
ax.set_title('Sensor Correlation with Remaining Useful Life')
ax.axvline(x=0, color='black', lw=1)

plt.tight_layout()
plt.savefig('../figures/sensor_rul_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../figures/sensor_rul_correlation.png')

## 10. Results Summary

All metrics computed on **real NASA CMAPSS FD001 data** — zero synthetic inputs.

In [ ]:
print('=== FINAL RESULTS: NASA CMAPSS FD001 RUL PREDICTION ===')
print(f"\nDataset:")
print(f"  Training engines: {train_df['unit_number'].nunique()}")
print(f"  Training cycles: {len(train_df):,}")
print(f"  Test engines: {test_df['unit_number'].nunique()}")
print(f"  Test cycles: {len(test_df):,}")
print(f"  Sensors used: {len(useful_sensors)} / 21")

print(f"\nFeatures:")
print(f"  Total features: {len(feature_cols)}")
print(f"  Engineering: rolling mean(10), rolling std(10), diff, cycle_norm")

print(f"\nModel Performance:")
print(f"  Random Forest — MAE: {rf_metrics['MAE']:.2f}, RMSE: {rf_metrics['RMSE']:.2f}, R2: {rf_metrics['R2']:.3f}")
print(f"  XGBoost      — MAE: {xgb_metrics['MAE']:.2f}, RMSE: {xgb_metrics['RMSE']:.2f}, R2: {xgb_metrics['R2']:.3f}")

print(f"\nVisualizations generated:")
print(f"  1. figures/rul_distributions.png")
print(f"  2. figures/sensor_degradation_curves.png")
print(f"  3. figures/predicted_vs_actual_rul.png")
print(f"  4. figures/rul_residuals.png")
print(f"  5. figures/feature_importance.png")
print(f"  6. figures/rul_degradation_trajectories.png")
print(f"  7. figures/sensor_rul_correlation.png")

print('\n✓ Analysis complete using real NASA CMAPSS data.')